In [ ]:
# Step 1: Now we are gonna analyze inactive users. 

import pandas as pd
import numpy as np

# Load Inactive sheet
df3 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Inactive_Users"
)
print(f"Raw shape: {df3.shape}")

# Drop completely empty rows
df3 = df3.dropna(how="all")
print(f"Shape after dropping empty rows: {df3.shape}")

# Rename columns to standardised names 
df3 = df3.rename(columns={
    "Baseline_Date":                 "BL_Entry",
    "HbA1c_baseline":                "HbA1c_BL",
    "BMI_baseline":                  "BMI_BL",
    "Waist_baseline":                "Waist_BL",
    "HDL CHOLESTEROL_baseline":      "HDL_BL",
    "SERUM TRIGLYCERIDES_baseline":  "TGL_BL",
    "BP Diastolic_baseline":         "BP_Dia_BL",
    "BP Systolic_baseline":          "BP_Sys_BL",
    "Serum Cholesterol_baseline":    "Serum_Cholesterol_BL",
    "FollowUpDate":                  "FU_Entry",
    "HbA1c_followup":                "HbA1c_FU",
    "BMI_followup":                  "BMI_FU",
    "Waist_followup":                "Waist_FU",
    "HDL CHOLESTEROL_followup":      "HDL_FU",
    "SERUM TRIGLYCERIDES_followup":  "TGL_FU",
    "BP Diastolic_followup":         "BP_Dia_Fu",
    "BP Systolic_followup":          "BP_Sys_FU",
    "Serum Cholesterol_followup":    "Serum_Cholesterol_FU",
})
print(f"\nColumns after rename: {list(df3.columns)}")

# Replace 0 with NaN for lab columns (0 not clinically valid)
zero_cols = [
    "TGL_BL", "TGL_FU",
    "Serum_Cholesterol_BL", "Serum_Cholesterol_FU",
    "HDL_BL", "HDL_FU"
]
for col in zero_cols:
    if col in df3.columns:
        df3[col] = df3[col].replace(0, np.nan)

# Force Diab_Duration to numeric first (mixed types)
df3["Diab_Duration"] = pd.to_numeric(df3["Diab_Duration"], errors="coerce")

# Coerce numeric columns — only what exists in this sheet
numeric_cols = [
    "AGE", "AOS", "Diab_Duration",
    "HbA1c_BL",  "BMI_BL",  "Waist_BL",
    "HDL_BL",    "TGL_BL",  "BP_Dia_BL",
    "BP_Sys_BL", "Serum_Cholesterol_BL",
    "HbA1c_FU",  "BMI_FU",  "Waist_FU",
    "HDL_FU",    "TGL_FU",  "BP_Dia_Fu",
    "BP_Sys_FU", "Serum_Cholesterol_FU",
]
for col in numeric_cols:
    if col in df3.columns:
        df3[col] = pd.to_numeric(df3[col], errors="coerce")

# Clean string columns only 
replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]
for col in df3.select_dtypes(include="object").columns:
    df3[col] = df3[col].str.strip()
    df3[col] = df3[col].replace(replace_vals, np.nan)

# NaN overview 
print(f"\nTotal NaN count: {df3.isna().sum().sum()}")
print("\nNaN per column:")
print(df3.isna().sum().to_string())

# 1. Kuppuswamy Occupation 
kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}

df3["Kupp_Occupation"] = df3["Occupation"].map(kupp_map)

n_before = len(df3)
df3 = df3.dropna(subset=["Kupp_Occupation"])
print(f"\nRemoved (missing/unmapped occupation): {n_before - len(df3)} rows")
print(f"Remaining: {len(df3)} rows")

# 2. Gender 
# Raw data uses M/F
df3["Gender"] = df3["Gender"].map({"M": "Male", "F": "Female"})
df3["Gender_Code"] = df3["Gender"].map({"Male": 1, "Female": 2})

# 3. Age Group 
# 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60
# Ref: Sathish et al., Indian J Med Res, 2010
def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

df3["Age_Group"] = df3["AGE"].apply(age_group)
age_labels = {0: "<30", 1: "30–39", 2: "40–49", 3: "50–59", 4: "≥60"}

# NOTE: No Engagement columns in Inactive sheet 
# Engagement_category, Credit_Score, VisitCount do not exist here
# Steps 7 & 8 will not apply to Inactive users

# Final verification 
print(f"\nAGE   — NaN: {df3['AGE'].isna().sum()} / {len(df3)}")
print(f"AOS   — NaN: {df3['AOS'].isna().sum()} / {len(df3)}")
print(f"Diab_Duration — NaN: {df3['Diab_Duration'].isna().sum()} / {len(df3)}")
print(f"\nFinal shape: {df3.shape}")
print("\nCoding reference:")
print("  Gender_Code         : 1=Male | 2=Female")
print("  Age_Group           : 0=<30 | 1=30–39 | 2=40–49 | 3=50–59 | 4=≥60")
print("  Kupp_Occupation     : 1–10 (Kuppuswamy scale)")
print("  NOTE: No Engagement_category / Credit_Score / VisitCount in this sheet")

print(df3[["MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
           "AOS", "Diab_Duration",
           "Occupation", "Kupp_Occupation"]].head(10).to_string())

In [ ]:
# Step 2: Missingness Check — Inactive Users; like before!!!

outcome_vars = {
    "HbA1c":             ("HbA1c_BL",  "HbA1c_FU"),
    "BMI":               ("BMI_BL",    "BMI_FU"),
    "HDL":               ("HDL_BL",    "HDL_FU"),
    "TGL":               ("TGL_BL",    "TGL_FU"),
    "Serum_Cholesterol": ("Serum_Cholesterol_BL", "Serum_Cholesterol_FU"),
}

THRESHOLD = 0.20
n_total   = len(df3)

print("=" * 60)
print(f"MISSINGNESS REPORT — INACTIVE  (n={n_total}, threshold={int(THRESHOLD*100)}%)")
print("=" * 60)
print(f"{'Variable':<22} {'BL Missing':>12} {'FU Missing':>12} {'Include?':>10}")
print("-" * 60)

include_vars = []
exclude_vars = []

for name, (bl_col, fu_col) in outcome_vars.items():
    bl_miss = df3[bl_col].isna().sum() / n_total if bl_col in df3.columns else 1.0
    fu_miss = df3[fu_col].isna().sum() / n_total if fu_col in df3.columns else 1.0
    worst   = max(bl_miss, fu_miss)
    flag    = "✓ YES" if worst < THRESHOLD else "✗ EXCLUDE"
    print(f"{name:<22} {bl_miss*100:>10.1f}%  {fu_miss*100:>10.1f}%  {flag:>10}")
    if worst < THRESHOLD:
        include_vars.append(name)
    else:
        exclude_vars.append(name)

print("=" * 60)
print(f"\n✓ INCLUDED ({len(include_vars)}): {include_vars}")
print(f"✗ EXCLUDED ({len(exclude_vars)}): {exclude_vars}")

In [ ]:
# Step 3: Finalise dataframe & calculate deltas for inactive users

keep_cols = [
    "MRNO", "Gender", "Gender_Code",
    "AGE", "Age_Group",
    "AOS", "Diab_Duration",
    "Occupation", "Kupp_Occupation",
    # Clinical outcomes
    "HbA1c_BL", "HbA1c_FU",
    "BMI_BL",   "BMI_FU",
]

df3 = df3[[c for c in keep_cols if c in df3.columns]].copy()
print(f"Columns kept: {df3.shape[1]}")
print(f"Columns: {list(df3.columns)}")

# Calculate deltas (FU - BL) 
# Both HbA1c and BMI: negative delta = improvement
df3["Delta_HbA1c"] = df3["HbA1c_FU"] - df3["HbA1c_BL"]
df3["Delta_BMI"]   = df3["BMI_FU"]   - df3["BMI_BL"]

# Drop rows missing BL+FU for both outcomes 
n_before = len(df3)

has_any_complete = (
    (df3["HbA1c_BL"].notna() & df3["HbA1c_FU"].notna()) |
    (df3["BMI_BL"].notna()   & df3["BMI_FU"].notna())
)

df3 = df3[has_any_complete].copy()
print(f"\nRows dropped (no complete BL+FU for any outcome): {n_before - len(df3)}")
print(f"Final analysis n: {len(df3)}")

# SES and Age Group helper columns
df3["SES_Group"] = pd.cut(
    df3["Kupp_Occupation"],
    bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df3["SES_Group_str"] = df3["SES_Group"].astype(str)
df3["Age_Group_str"] = df3["Age_Group"].map(age_labels)

# Delta summary 
delta_cols = ["Delta_HbA1c", "Delta_BMI"]

print("\n" + "=" * 60)
print("DELTA SUMMARY (FU - BL)")
print("  HbA1c : negative delta = improvement")
print("  BMI   : negative delta = improvement")
print("=" * 60)
print(df3[delta_cols].describe().round(3).to_string())

print("\nMissing delta values:")
for col in delta_cols:
    n_miss = df3[col].isna().sum()
    print(f"  {col:<20} {n_miss} missing ({n_miss/len(df3)*100:.1f}%)")

print(df3[["MRNO", "Gender", "Gender_Code", "AGE", "Age_Group",
           "AOS", "Diab_Duration", "Kupp_Occupation",
           "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
           "BMI_BL",   "BMI_FU",   "Delta_BMI"]].head(10).to_string())

In [ ]:
# Step 4: Descriptive Profile for inactive users 

print("=" * 60)
print("DESCRIPTIVE PROFILE — INACTIVE USERS")
print("=" * 60)

# Continuous variables 
continuous = {
    "Age (years)":               "AGE",
    "Age of Onset (years)":      "AOS",
    "Diabetes Duration (years)": "Diab_Duration",
    "Kuppuswamy SES Score":      "Kupp_Occupation",
    "HbA1c BL (%)":             "HbA1c_BL",
    "BMI BL (kg/m²)":           "BMI_BL",
}

print(f"\n{'Variable':<30} {'n':>6} {'Mean':>8} {'SD':>8} {'Min':>8} {'Max':>8}")
print("-" * 72)
for label, col in continuous.items():
    if col in df3.columns:
        n    = df3[col].notna().sum()
        mean = df3[col].mean()
        sd   = df3[col].std()
        mn   = df3[col].min()
        mx   = df3[col].max()
        print(f"{label:<30} {n:>6} {mean:>8.2f} {sd:>8.2f} {mn:>8.2f} {mx:>8.2f}")

# Categorical variables 
print("\n" + "=" * 60)
print("CATEGORICAL VARIABLES")
print("=" * 60)

# Gender
print("\nGender:")
for val, count in df3["Gender"].value_counts(dropna=True).items():
    print(f"  {val:<15} n={count:>4}  ({count/len(df3)*100:.1f}%)")

# Age Group
print("\nAge Group:")
for val, count in df3["Age_Group"].value_counts(dropna=True).sort_index().items():
    print(f"  {age_labels[val]:<15} n={count:>4}  ({count/len(df3)*100:.1f}%)")

# SES — granular
print("\nSES Score — Granular (Kuppuswamy 1–10):")
kupp_labels = {
    1:  "1  — Unskilled/Housewife",
    2:  "2  — Daily wages",
    3:  "3  — Driver/Courier",
    4:  "4  — Private Sector",
    5:  "5  — Farmer/Agriculture",
    6:  "6  — Business/Self-Employed",
    7:  "7  — Clerk/Government",
    8:  "8  — Bank/Supervisor/Armed Forces",
    9:  "9  — Doctor/Engineer/Advocate",
    10: "10 — Politician/Manager/Senior Govt",
}
for val, count in df3["Kupp_Occupation"].value_counts(dropna=True).sort_index().items():
    label = kupp_labels.get(int(val), str(val))
    print(f"  {label:<40} n={count:>4}  ({count/len(df3)*100:.1f}%)")

# SES — binned
print("\nSES Group — Binned:")
for val, count in df3["SES_Group"].value_counts(dropna=True).sort_index().items():
    print(f"  {str(val):<15} n={count:>4}  ({count/len(df3)*100:.1f}%)")

In [ ]:
# Step 5: Clinical Outcomes BL → FU for inactive users 
from scipy import stats
import numpy as np

def cohens_d_paired(a, b):
    diff = a - b
    return diff.mean() / diff.std()

print("=" * 70)
print("STEP 5: CLINICAL OUTCOMES — INACTIVE USERS (BL → FU)")
print("=" * 70)

outcomes = {
    "HbA1c": ("HbA1c_BL", "HbA1c_FU", "Delta_HbA1c", "negative"),
    "BMI":   ("BMI_BL",   "BMI_FU",   "Delta_BMI",   "negative"),
}

results3 = []

for name, (bl_col, fu_col, delta_col, direction) in outcomes.items():
    sub = df3[[bl_col, fu_col, delta_col]].dropna()
    n   = len(sub)

    if n < 3:
        print(f"\n{name}: insufficient data (n={n}), skipping.")
        continue

    bl_mean = sub[bl_col].mean()
    bl_sd   = sub[bl_col].std()
    fu_mean = sub[fu_col].mean()
    fu_sd   = sub[fu_col].std()
    delta   = sub[delta_col].mean()

    # Normality check 
    delta_vals = sub[delta_col]
    if len(delta_vals) > 5000:
        delta_sample = delta_vals.sample(5000, random_state=42)
    else:
        delta_sample = delta_vals

    shapiro_stat, shapiro_p = stats.shapiro(delta_sample)
    normal = shapiro_p > 0.05

    # Paired test 
    if normal:
        test_name = "Paired t-test"
        t_stat, p_val = stats.ttest_rel(sub[bl_col], sub[fu_col])
    else:
        test_name = "Wilcoxon"
        t_stat, p_val = stats.wilcoxon(sub[bl_col], sub[fu_col])

    cd  = cohens_d_paired(sub[bl_col], sub[fu_col])
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"

    if direction == "negative":
        improved = "✓ improved" if delta < 0 else "✗ worsened"
    else:
        improved = "✓ improved" if delta > 0 else "✗ worsened"

    results3.append({
        "Variable":   name,
        "n":          n,
        "BL Mean±SD": f"{bl_mean:.2f} ± {bl_sd:.2f}",
        "FU Mean±SD": f"{fu_mean:.2f} ± {fu_sd:.2f}",
        "Delta":      round(delta, 3),
        "Test":       test_name,
        "Stat":       round(t_stat, 3),
        "p_value":    round(p_val, 4),
        "Sig":        sig,
        "Cohens_D":   round(abs(cd), 3),
        "Direction":  improved,
    })

    print(f"\n{'─' * 70}")
    print(f"{name} {sig}  |  {improved}  |  n={n}")
    print(f"  BL:        {bl_mean:.2f} ± {bl_sd:.2f}")
    print(f"  FU:        {fu_mean:.2f} ± {fu_sd:.2f}")
    print(f"  Delta:     {delta:.3f}")
    print(f"  Test:      {test_name} | stat={t_stat:.3f} | p={p_val:.4f} {sig}")
    print(f"  Cohen's D: {abs(cd):.3f}")
    print(f"  Normality: SW p={shapiro_p:.3f} ({'normal' if normal else 'non-normal'})")

# Summary table 
print(f"\n{'=' * 70}")
print("SUMMARY TABLE — INACTIVE USERS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'BL Mean±SD':<18} {'FU Mean±SD':<18} "
      f"{'Delta':>8} {'p-value':>8} {'Sig':>5} {'CohenD':>8} {'Result'}")
print("-" * 105)
for r in results3:
    cd = r["Cohens_D"]
    pv = r["p_value"]
    print(f"{r['Variable']:<22} {r['n']:>5} {r['BL Mean±SD']:<18} {r['FU Mean±SD']:<18} "
          f"{r['Delta']:>8} {pv:>8} {r['Sig']:>5} {cd:>8} {r['Direction']}")

In [ ]:
# Step 6: Does SES predict clinical improvement? — inactive users 
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("STEP 6: SES vs CLINICAL IMPROVEMENT — INACTIVE USERS")
print("=" * 70)

delta_outcomes = {
    "HbA1c": ("Delta_HbA1c", "negative"),
    "BMI":   ("Delta_BMI",   "negative"),
}

ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]

# Manual Dunn's post-hoc 
def dunn_posthoc(data, group_col, val_col, groups):
    from scipy.stats import rankdata
    all_vals   = data[val_col].values
    all_groups = data[group_col].values
    n_total    = len(all_vals)
    ranks      = rankdata(all_vals)
    pairs      = list(combinations(groups, 2))
    results    = {}
    for g1, g2 in pairs:
        idx1 = all_groups == g1
        idx2 = all_groups == g2
        n1, n2 = idx1.sum(), idx2.sum()
        r1, r2 = ranks[idx1].mean(), ranks[idx2].mean()
        se = np.sqrt((n_total * (n_total + 1) / 12.0) * (1.0 / n1 + 1.0 / n2))
        z  = (r1 - r2) / se
        p  = 2 * stats.norm.sf(abs(z))
        results[(g1, g2)] = p
    n_pairs   = len(pairs)
    corrected = {k: min(v * n_pairs, 1.0) for k, v in results.items()}
    return corrected

step6_results3 = []

for name, (delta_col, direction) in delta_outcomes.items():

    sub = df3[["SES_Group_str", "Kupp_Occupation", delta_col]].dropna().copy()
    n   = len(sub)

    print(f"\n{'─' * 70}")
    print(f"{name}  |  n={n}")

    # Group means 
    print(f"\n  Group means:")
    for grp in ses_order:
        grp_data = sub[sub["SES_Group_str"] == grp][delta_col]
        if len(grp_data) > 0:
            print(f"    {grp:<18} n={len(grp_data):>4}  "
                  f"mean={grp_data.mean():>7.3f}  sd={grp_data.std():>7.3f}")

    # Kruskal-Wallis 
    groups = [
        sub[sub["SES_Group_str"] == g][delta_col].dropna().values
        for g in ses_order
    ]
    groups = [g for g in groups if len(g) >= 3]

    if len(groups) < 2:
        print("  Insufficient groups for Kruskal-Wallis, skipping.")
        continue

    kw_stat, kw_p = stats.kruskal(*groups)
    kw_sig = "***" if kw_p < 0.001 else "**" if kw_p < 0.01 else "*" if kw_p < 0.05 else "ns"
    print(f"\n  Kruskal-Wallis: H={kw_stat:.3f}, p={kw_p:.4f} {kw_sig}")

    # Dunn's post-hoc (if significant) 
    if kw_p < 0.05:
        print("  Dunn's post-hoc (Bonferroni corrected):")
        dunn_results = dunn_posthoc(
            sub, group_col="SES_Group_str", val_col=delta_col, groups=ses_order
        )
        for (g1, g2), p in dunn_results.items():
            sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            print(f"    {str(g1):<18} vs {str(g2):<18} p={p:.4f} {sig}")
    else:
        print("  No significant difference — post-hoc not run.")

    # Spearman correlation 
    spear_r, spear_p = stats.spearmanr(sub["Kupp_Occupation"], sub[delta_col])
    spear_sig = "***" if spear_p < 0.001 else "**" if spear_p < 0.01 else "*" if spear_p < 0.05 else "ns"
    print(f"\n  Spearman (Kupp score vs delta): r={spear_r:.3f}, "
          f"p={spear_p:.4f} {spear_sig}")

    step6_results3.append({
        "Variable":     name,
        "n":            n,
        "KW_H":         round(kw_stat, 3),
        "KW_p":         round(kw_p, 4),
        "KW_sig":       kw_sig,
        "Spearman_r":   round(spear_r, 3),
        "Spearman_p":   round(spear_p, 4),
        "Spearman_sig": spear_sig,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("STEP 6 SUMMARY TABLE — INACTIVE USERS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'n':>5} {'KW H':>8} {'KW p':>8} {'Sig':>5} "
      f"{'Spearman r':>12} {'Spearman p':>12} {'Sig':>5}")
print("-" * 80)
for r in step6_results3:
    print(f"{r['Variable']:<22} {r['n']:>5} {r['KW_H']:>8} {r['KW_p']:>8} "
          f"{r['KW_sig']:>5} {r['Spearman_r']:>12} {r['Spearman_p']:>12} "
          f"{r['Spearman_sig']:>5}")

# Box plots 
colors = ["#d9534f", "#f0ad4e", "#5cb85c"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes = axes.flatten()

for i, (name, (delta_col, direction)) in enumerate(delta_outcomes.items()):
    sub = df3[["SES_Group_str", delta_col]].dropna().copy()
    data_by_group = [
        sub[sub["SES_Group_str"] == g][delta_col].values
        for g in ses_order
    ]
    ax = axes[i]
    bp = ax.boxplot(data_by_group, tick_labels=ses_order, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_title(f"{name} — Delta by SES Group", fontweight="bold")
    ax.set_xlabel("SES Group")
    ax.set_ylabel(f"Delta {name} (FU - BL)")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Clinical Improvement by SES Group — Inactive Users",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step6_inactive_ses_vs_deltas.png", dpi=150, bbox_inches="tight")
plt.close()